<a href="https://colab.research.google.com/github/G1G1KO/Breast-Cancer-Detector/blob/main/BreastCancerClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

class DeepNeuralNetwork:
    def __init__(self, input_size, h1_size, h2_size, output_size, lr=0.1):
        """
        Initialize weights and hyperparameters.
        """
        np.random.seed(42)
        self.lr = lr

        # Weight initialization (Input -> H1 -> H2 -> Output)
        self.W1 = np.random.randn(input_size + 1, h1_size) * 0.1
        self.W2 = np.random.randn(h1_size + 1, h2_size) * 0.1
        self.W3 = np.random.randn(h2_size + 1, output_size) * 0.1

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def _sigmoid_derivative(self, a):
        return a * (1 - a)

    def _add_bias(self, X):
        return np.insert(X, 0, 1, axis=1)

    def forward(self, X):
        """
        Perform a forward pass through the network.
        """
        # Input to Hidden 1
        self.input_with_bias = self._add_bias(X)
        self.z1 = np.dot(self.input_with_bias, self.W1)
        self.a1 = self._sigmoid(self.z1)

        # Hidden 1 to Hidden 2
        self.a1_with_bias = self._add_bias(self.a1)
        self.z2 = np.dot(self.a1_with_bias, self.W2)
        self.a2 = self._sigmoid(self.z2)

        # Hidden 2 to Output
        self.a2_with_bias = self._add_bias(self.a2)
        self.z3 = np.dot(self.a2_with_bias, self.W3)
        self.a3 = self._sigmoid(self.z3)

        return self.a3

    def train(self, X, y, epochs=5000):
        """
        Train the network using backpropagation.
        """
        for epoch in range(epochs):
            # 1. Forward Pass
            output = self.forward(X)

            # 2. Compute Errors
            error = output - y

            # 3. Backpropagation (Chain Rule)
            d_out = error * self._sigmoid_derivative(output)

            error_h2 = np.dot(d_out, self.W3[1:].T)
            d_h2 = error_h2 * self._sigmoid_derivative(self.a2)

            error_h1 = np.dot(d_h2, self.W2[1:].T)
            d_h1 = error_h1 * self._sigmoid_derivative(self.a1)

            # 4. Update Weights (Gradient Descent)
            n = len(X)
            self.W3 -= self.lr * np.dot(self.a2_with_bias.T, d_out) / n
            self.W2 -= self.lr * np.dot(self.a1_with_bias.T, d_h2) / n
            self.W1 -= self.lr * np.dot(self.input_with_bias.T, d_h1) / n

            if epoch % 1000 == 0:
                loss = np.mean(0.5 * (error**2))
                print(f"Epoch {epoch}: Loss = {loss:.4f}")

    def predict(self, X):
        """
        Predict binary labels (0 or 1).
        """
        probabilities = self.forward(X)
        return np.round(probabilities)

# --- Execution Section ---

# Load and scale data
data = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target.reshape(-1, 1), test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train model
model = DeepNeuralNetwork(input_size=30, h1_size=16, h2_size=8, output_size=1)
model.train(X_train_scaled, y_train, epochs=5000)

# Evaluate model
test_predictions = model.predict(X_test_scaled)
accuracy = np.mean(test_predictions == y_test) * 100
print(f"\nFinal Test Accuracy: {accuracy:.2f}%")

Epoch 0: Loss = 0.1358
Epoch 1000: Loss = 0.1160
Epoch 2000: Loss = 0.1089
Epoch 3000: Loss = 0.0442
Epoch 4000: Loss = 0.0176

Final Test Accuracy: 98.25%
